<a href="https://colab.research.google.com/github/Priyanshu27083/DL_practice/blob/main/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Guided Back Propagation**

In [ ]:
import numpy as np

# -----------------------------
# 1. Define inputs (pixels)
# -----------------------------
x = np.array([1.0, -2.0, 3.0])   # input pixels

# -----------------------------
# 2. Define weights & bias
# -----------------------------
w = np.array([0.5, -1.0, 2.0])
b = 1.0

# -----------------------------
# 3. Forward Pass
# -----------------------------
# Linear combination
z = np.dot(w, x) + b
print("Linear output (z):", z)

# ReLU activation
y = max(0, z)
print("Activated output (y):", y)

# -----------------------------
# 4. Normal Backpropagation
# -----------------------------
# derivative of ReLU
dz = 1.0 if z > 0 else 0.0

# gradient w.r.t inputs
grad_normal = dz * w
print("Normal Backprop Gradients:", grad_normal)

# -----------------------------
# 5. Guided Backpropagation
# -----------------------------
grad_guided = grad_normal.copy()

# Apply guided rule:
# keep only positive gradients AND positive forward activation
for i in range(len(grad_guided)):
    if grad_guided[i] < 0 or z <= 0:
        grad_guided[i] = 0

print("Guided Backprop Gradients:", grad_guided)

# -----------------------------
# 6. Interpretation
# -----------------------------
for i, g in enumerate(grad_guided):
    if g > 0:
        print(f"Pixel x{i+1} influences output (importance = {g})")
    else:
        print(f"Pixel x{i+1} does NOT influence output")

Linear output (z): 9.5
Activated output (y): 9.5
Normal Backprop Gradients: [ 0.5 -1.   2. ]
Guided Backprop Gradients: [0.5 0.  2. ]
Pixel x1 influences output (importance = 0.5)
Pixel x2 does NOT influence output
Pixel x3 influences output (importance = 2.0)


In [ ]:
seed = 'd'
generated = []

for _ in range(5):
    x = np.eye(len(vocab))[char2idx[seed[-1]]].reshape(1, 1, len(vocab))
    pred = model.predict(x, verbose=0)
    next_char = idx2char[np.argmax(pred)]

    if next_char == '<stop>':
        break

    generated.append(next_char)
    seed += next_char

print("Generated:", ''.join(generated))

NameError: name 'vocab' is not defined

In [ ]:
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense
import numpy as np

# Define vocabulary
vocab = ['d', 'e', 'p', '<stop>']
char2idx = {c: i for i, c in enumerate(vocab)}
idx2char = {i: c for c, i in char2idx.items()}

# Example input sequence: "dep"
sequence = ['d', 'e', 'p']
target = ['e', 'p', '<stop>']  # Next characters

# One-hot encode input and target
X = np.eye(len(vocab))[[char2idx[c] for c in sequence]].reshape(len(sequence), 1, len(vocab))
y = np.eye(len(vocab))[[char2idx[c] for c in target]]
# Build RNN model
model = Sequential([
    SimpleRNN(8, input_shape=(1, len(vocab))),
    Dense(len(vocab), activation='softmax')  # softmax output
])

model.compile(loss='categorical_crossentropy', optimizer='adam')  # cross-entropy loss

# Train the model
model.fit(X, y, epochs=200, verbose=0)

# Predict next character after 'p'
test = np.eye(len(vocab))[char2idx['p']].reshape(1, 1, len(vocab))
pred = model.predict(test)
print("Next char prediction:", idx2char[np.argmax(pred)])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
Next char prediction: <stop>


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Flatten
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# 1. Prepare the data
reviews = ["good movie", "bad movie", "excellent film", "terrible acting"]
labels = np.array([1, 0, 1, 0]) # 1 for positive, 0 for negative

# Tokenize the text
tokenizer = Tokenizer(num_words=None) # Consider all words
tokenizer.fit_on_texts(reviews)
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1 # +1 for the padding token

# Convert text to sequences of integers
sequences = tokenizer.texts_to_sequences(reviews)

# Pad sequences to have the same length
max_length = max(len(seq) for seq in sequences)
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

# 2. Define the Keras model
embedding_dim = 10
model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length), # Embed word indices into vectors
    SimpleRNN(8),                                              # Simple RNN layer with 8 hidden units
    Dense(1, activation='sigmoid')                             # Output layer with sigmoid for binary classification
])

# 3. Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 4. Train the model (for a few epochs for demonstration)
model.fit(padded_sequences, labels, epochs=10, verbose=0)

# 5. Predict the sentiment of the first review
test_review = ["amazing movie"]
test_sequence = tokenizer.texts_to_sequences(test_review)
padded_test_sequence = pad_sequences(test_sequence, maxlen=max_length, padding='post')
prediction = model.predict(padded_test_sequence)
sentiment = "Positive" if prediction[0] > 0.5 else "Negative"

# 6. Print the result
print(f"Review: '{test_review[0]}', Predicted Sentiment: {sentiment} (Confidence: {prediction[0][0]:.2f})")



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 245ms/step
Review: 'amazing movie', Predicted Sentiment: Positive (Confidence: 0.51)


In [ ]:
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense
import numpy as np

# Define vocabulary
vocab = ['d', 'e', 'p', '<stop>']
char2idx = {c: i for i, c in enumerate(vocab)}
idx2char = {i: c for c, i in char2idx.items()}

# Example input sequence: "dep"
sequence = ['d', 'e', 'p']
target = ['e', 'p', '<stop>']  # Next characters

# One-hot encode input and target
X = np.eye(len(vocab))[[char2idx[c] for c in sequence]].reshape(len(sequence), 1, len(vocab))
y = np.eye(len(vocab))[[char2idx[c] for c in target]]

# Build RNN model
model = Sequential([
    SimpleRNN(8, input_shape=(1, len(vocab))),
    Dense(len(vocab), activation='softmax')  # softmax output
])
model.compile(loss='categorical_crossentropy', optimizer='adam')  # cross-entropy loss

# Train the model
model.fit(X, y, epochs=200, verbose=0)

# Predict next character after 'p'
test = np.eye(len(vocab))[char2idx['e']].reshape(1, 1, len(vocab))
pred = model.predict(test)
print("Next char prediction:", idx2char[np.argmax(pred)])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step
Next char prediction: p


**Predict Next Word**

In [ ]:
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense
import numpy as np

# Define a small vocabulary of words
vocab = ['deep', 'learning', 'is', '<stop>']
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

# Input sequence of words
sequence = ['deep', 'learning', 'is']
target = ['learning', 'is', '<stop>']  # Next words

# One-hot encode words
X = np.eye(len(vocab))[[word2idx[w] for w in sequence]].reshape(len(sequence), 1, len(vocab))
y = np.eye(len(vocab))[[word2idx[w] for w in target]]

# Build a simple RNN model
model = Sequential([
    SimpleRNN(8, input_shape=(1, len(vocab))),
    Dense(len(vocab), activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam')

# Train the model
model.fit(X, y, epochs=300, verbose=0)

# Predict the next word after 'is'
test = np.eye(len(vocab))[word2idx['is']].reshape(1, 1, len(vocab))
pred = model.predict(test)
print("Next word prediction:", idx2word[np.argmax(pred)])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step
Next word prediction: <stop>


In [ ]:
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense, Embedding
import numpy as np

# Sample vocabulary
vocab = {'this': 0, 'movie': 1, 'is': 2, 'great': 3, 'bad': 4, 'terrible': 5}
vocab_size = len(vocab)

# Sample data: [this movie is great] -> positive (1), [this movie is terrible] -> negative (0)
X = np.array([
    [0, 1, 2, 3],  # "this movie is great"
    [0, 1, 2, 5]   # "this movie is terrible"
])
y = np.array([1, 0])  # Sentiment labels

# Model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=8, input_length=4),
    SimpleRNN(16),  # Outputs only at the last timestep
    Dense(1, activation='sigmoid')  # Binary classification
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train
model.fit(X, y, epochs=100, verbose=0)

# Test
test = np.array([[0, 1, 2, 5]])  # "this movie is bad"
pred = model.predict(test)
print("Predicted sentiment:", "Positive" if pred[0][0] > 0.5 else "Negative")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
Predicted sentiment: Negative


**BERT**

In [ ]:
!pip install --upgrade transformers
# Step 0: Clean install (important)
!pip uninstall -y transformers torch torchvision torchaudio -q
!pip install -q transformers torch

# Step 1: Import
from transformers import pipeline

# Step 2: Load sentiment pipeline (BERT-based model)
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# Step 3: Input text
text = "I love this product!"

# Step 4: Predict
result = classifier(text)[0]

# Step 5: Convert to binary (1 = positive, 0 = negative)
label = result['label']
binary_output = 1 if label == "POSITIVE" else 0

# Step 6: Print results
print("Text:", text)
print("Model Output:", result)
print("Binary Sentiment (1=Positive, 0=Negative):", binary_output)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 29.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Text: I love this product!
Model Output: {'label': 'POSITIVE', 'score': 0.9998855590820312}
Binary Sentiment (1=Positive, 0=Negative): 1


**GAN**

In [ ]:
import numpy as np
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU, Reshape, Flatten
from tensorflow.keras.optimizers import Adam

# Load and normalize MNIST data
(X_train, _), _ = mnist.load_data()
X_train = (X_train - 127.5) / 127.5  # Scale to [-1, 1]
X_train = X_train.reshape(-1, 784)

# Optimizer
opt = Adam(0.0002, 0.5)

# --- Generator function ---
def build_generator():
    model = Sequential([
        Dense(128, input_dim=100),
        LeakyReLU(0.2),
        Dense(784, activation='tanh'),  # Output shape: 28x28 = 784
    ])
    return model

# --- Discriminator function ---
def build_discriminator():
    model = Sequential([
        Dense(128, input_shape=(784,)),
        LeakyReLU(0.2),
        Dense(1, activation='sigmoid')  # Binary output: real (1) or fake (0)
    ])
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Build models
generator = build_generator()
discriminator = build_discriminator()

# --- Train GAN ---
noise = np.random.normal(0, 1, (128, 100))          # Random noise
fake_images = generator.predict(noise)              # Generate fake images

real_images = X_train[np.random.randint(0, X_train.shape[0], 128)]  # Sample real images

# Labels for training
real_labels = np.ones((128, 1))
fake_labels = np.zeros((128, 1))

# Train discriminator
d_loss_real = discriminator.train_on_batch(real_images, real_labels)
d_loss_fake = discriminator.train_on_batch(fake_images, fake_labels)

# Print discriminator performance
print("Discriminator accuracy on real:", d_loss_real[1]*100)
print("Discriminator accuracy on fake:", d_loss_fake[1]*100)


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step 
Discriminator accuracy on real: 0.0
Discriminator accuracy on fake: 28.125
